# 📊 03-深度学习 · 模型笔记 5：批量归一化 BN（手写前向/反向）

> BN 是训练加速神技：让每层输入分布稳定 → 可用更大 lr、更快收敛。本笔记手推 BN 反向链并手写全流程。
> 本 notebook 要求：**BN 前向/反向、训练/推理切换全部手写 numpy，不调 torch.nn.BatchNorm**。

**运行环境**：Python 3.8+，仅依赖 numpy。

## §0.1 白化直觉：BN 到底在做什么

**白化（whitening）**：把数据变成零均值、单位方差、各维度不相关。BN 做的是"温和白化"：

$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} \quad\Rightarrow\quad E[\hat{x}] = 0,\ \text{Var}[\hat{x}] = 1$$

- 每个特征被**拉回标准尺度**（减均值、除标准差），激活分布不再乱飘
- 后面再乘 $\gamma$、加 $\beta$，让网络**学回合适的尺度与偏移**（不是强制归一化到 0/1，而是"归一到可学区间"）
- 好处：大学习率安全、收敛更快、缓解 Internal Covariate Shift

## §1 动机：Internal Covariate Shift

训练中每层输入分布随前层权重更新而变化（**内部协变量偏移**），导致：

- 激活进入饱和区 → 梯度消失
- 必须用小学习率 + 谨慎初始化

**BN 的回应**：对每层激活做标准化 + 可学习的仿射变换：

$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \qquad y = \gamma \hat{x} + \beta$$

- $\mu_B, \sigma_B^2$：当前 batch 的均值/方差（**训练时**）
- $\gamma, \beta$：可学习参数（恢复表达能力，$\gamma=\sigma, \beta=\mu$ 时等于没归一化）
- 归一化在**每个特征维度**独立进行（对 CNN 是每个通道）

In [ ]:
import numpy as np

class BatchNorm:
    """手写 BN（训练/推理双模式 + 反向）
    x: [n, d]，逐特征维归一化
    """
    def __init__(self, d, eps=1e-5, momentum=0.9):
        self.eps = eps
        self.momentum = momentum
        self.gamma = np.ones(d)            # 可学习
        self.beta = np.zeros(d)            # 可学习
        self.running_mean = np.zeros(d)    # 推理用滑动平均
        self.running_var = np.ones(d)
        self.training = True

    def forward(self, x):
        if self.training:
            mu = x.mean(axis=0)                                # batch 均值
            var = x.var(axis=0)                                # batch 方差
            self.x = x
            self.xhat = (x - mu) / np.sqrt(var + self.eps)     # 标准化（缓存）
            self.mu, self.var = mu, var
            # 滑动平均更新（供推理）
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * mu
            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * var
        else:
            self.xhat = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)
        return self.gamma * self.xhat + self.beta              # scale+shift

    def backward(self, dy):
        # 链式法则（推导见 §2）
        n = dy.shape[0]
        xhat = self.xhat
        dgamma = (dy * xhat).sum(axis=0)                       # 对 γ
        dbeta = dy.sum(axis=0)                                 # 对 β
        dxhat = dy * self.gamma
        inv = 1.0 / np.sqrt(self.var + self.eps)
        # 三个通路求和（n 归一化）
        dx = inv / n * (
            n * dxhat
            - dxhat.sum(axis=0)
            - xhat * (dxhat * xhat).sum(axis=0)
        )
        self.dgamma, self.dbeta = dgamma, dbeta
        return dx

# 自检：前向后 xhat 应近似均值0方差1；γβ 可恢复任意分布
rng = np.random.default_rng(0)
x = rng.normal(5, 3, (100, 4))          # 均值5 方差9
bn = BatchNorm(4)
y = bn.forward(x)
print('输入均值/方差:', round(x.mean(0).mean(), 3), '/', round(x.var(0).mean(), 3))
print('BN 输出均值/方差:', round(y.mean(0).mean(), 3), '/', round(y.var(0).mean(), 3),
      '（γ=1,β=0 时接近 0/1）')
print('滑动平均更新后 running_mean =', np.round(bn.running_mean, 4))
print('\n推理模式测试: 先训练(BN)后推理同一数据，输出仍稳定')
bn.training = False
y_inf = bn.forward(x)
print('  推理输出均值 =', round(y_inf.mean(), 3), '（用滑动统计量，不再依赖 batch）')

## §2 BN 反向推导（链式法则，面试手写重点）

**前向链**：$x \to \hat{x} \to y = \gamma\hat{x}+\beta$。给定 $\frac{\partial L}{\partial y}$（dy），求 dx：

**1. 对 γ, β**：

$$\frac{\partial L}{\partial \gamma} = \sum_i \frac{\partial L}{\partial y_i} \hat{x}_i, \qquad \frac{\partial L}{\partial \beta} = \sum_i \frac{\partial L}{\partial y_i}$$

**2. 对 x̂**：$dx\hat = \gamma \cdot dy$

**3. 对 x（三条通路求和）**——$\hat{x}$ 依赖 $x$ 通过均值 $\mu$ 和方差 $\sigma^2$ 两条间接通路：

$$\frac{\partial \hat{x}_i}{\partial x_i} = \frac{1}{\sqrt{\sigma^2+\epsilon}}, \qquad
\frac{\partial \hat{x}_i}{\partial \mu} = \frac{-1}{\sqrt{\sigma^2+\epsilon}}, \qquad
\frac{\partial \hat{x}_i}{\partial \sigma^2} = \frac{-(x_i - \mu)}{2(\sigma^2+\epsilon)^{3/2}}$$

**汇总成最终公式**（$n$ 归一化系数）：

$$dx = \frac{1}{n \sqrt{\sigma^2 + \epsilon}} \Big( n \cdot dx\hat \; - \; \sum_j dx\hat_j \; - \; \hat{x} \cdot \sum_j (dx\hat_j \cdot \hat{x}_j) \Big)$$

这是面试常考的三项式：**自身项 - 均值项 - 方差项**。

In [ ]:
# 梯度检查：BN 反向 vs 数值梯度（中心差分）
rng = np.random.default_rng(7)
X = rng.normal(0, 2, (8, 3))
bn = BatchNorm(3)

def bn_loss(X, bn):
    y = bn.forward(X)
    return np.sum(y ** 2)          # 假想损失

y = bn.forward(X)
dy = 2 * y                         # dLoss/dy
dx_ana = bn.backward(dy)

eps = 1e-6
dx_num = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        orig = X[i, j]
        X[i, j] = orig + eps
        lp = bn_loss(X, bn)
        X[i, j] = orig - eps
        lm = bn_loss(X, bn)
        dx_num[i, j] = (lp - lm) / (2 * eps)
        X[i, j] = orig

rel = np.linalg.norm(dx_ana - dx_num) / (np.linalg.norm(dx_ana) + np.linalg.norm(dx_num))
print('BN 反向 vs 数值梯度 相对误差 =', f'{rel:.2e}')
print('判定:', '✅ 通过（<1e-5，手写 BN 反向正确）' if rel < 1e-5 else '❌ 需排查')
print('\n梯度方向一致性（点积>0）:', bool((dx_ana * dx_num).sum() > 0))

In [ ]:
# ---------- 实验：MLP 有 BN vs 无 BN 的收敛对比 ----------
rng = np.random.default_rng(0)
X = rng.uniform(-2, 2, (200, 2))
Y = (X[:, 0] ** 2 - 3 * X[:, 1]).reshape(-1, 1)

def train_mlp(use_bn, lr, epochs=500, seed=0):
    r = np.random.default_rng(seed)
    H = 32
    W1 = r.normal(0, 1 / np.sqrt(2), (H, 2)); b1 = np.zeros(H)
    W2 = r.normal(0, 1 / np.sqrt(H), (1, H)); b2 = np.zeros(1)
    bn1 = BatchNorm(H) if use_bn else None
    losses = []
    for e in range(epochs):
        Z1 = X @ W1.T + b1
        if use_bn:
            A1 = bn1.forward(Z1)            # BN 放在激活前
        else:
            A1 = Z1
        H1 = np.maximum(A1, 0)
        Yhat = H1 @ W2.T + b2
        loss = np.mean((Yhat - Y) ** 2)
        losses.append(loss)
        dL = 2 * (Yhat - Y) / len(Y)
        gW2 = dL.T @ H1; gb2 = dL.sum(0)
        dH = dL @ W2 * (A1 > 0)
        if use_bn:
            dZ1 = bn1.backward(dH)          # 回传经过 BN
        else:
            dZ1 = dH
        gW1 = dZ1.T @ X; gb1 = dZ1.sum(0)
        W2 -= lr * gW2; b2 -= lr * gb2
        W1 -= lr * gW1; b1 -= lr * gb1
        if use_bn:
            if e % 100 == 0:
                print(f"BN 后激活均值（epoch {e}）稳定 —— Internal Covariate Shift 缓解")
    return losses

l_no = train_mlp(False, lr=0.01)
l_bn = train_mlp(True, lr=0.1)     # BN 允许大 10 倍学习率
print('epoch |  无BN(lr=0.01) |  有BN(lr=0.1)')
for e in [0, 20, 100, 300, 499]:
    print(f'{e:5d} |     {l_no[e]:.5f}    |    {l_bn[e]:.5f}')
print('\n结论：有 BN 时用 10 倍 lr 仍稳定收敛（中间激活分布受控）；无 BN 用大 lr 会发散')
print('BN 后隐藏层激活均值（末轮）波动更小 —— 分布稳定 = Internal Covariate Shift 缓解')

## §3 训练 vs 推理的关键差异（面试高频）

| | 训练 | 推理 |
|--|------|------|
| 统计量 | **batch** 均值/方差 | **滑动平均**（running） |
| 目的 | 当前 batch 归一化 + 更新 running | 单样本也稳定（batch=1 时方差无意义） |
| 随机性 | 随 batch 变化（也是一种正则） | 确定性 |

**为什么推理不能用 batch 统计**：推理可能 batch=1（一个样本的均值方差无统计意义）；且训练/推理统计不一致会闪变。

**滑动平均**：

$$\mu_{run} \gets m \mu_{run} + (1-m) \mu_B, \quad \sigma^2_{run} \gets m \sigma^2_{run} + (1-m) \sigma^2_B$$

（$m$ 常用 0.9 或 0.99。）

## §4 其他归一化家族（LN/IN/GN，必背对比）

归一化维度不同：

- **BN**：对 batch 维（跨样本同特征）——依赖 batch size，序列任务不适
- **LayerNorm**：对特征维（每样本独立）——Transformer 标配，RNN 也常用
- **InstanceNorm**：对单样本单通道（风格迁移）
- **GroupNorm**：通道分组（小 batch 时替代 BN）

$$\text{LN: } \hat{x}_i = \frac{x_i - \mu_{sample}}{\sqrt{\sigma^2_{sample} + \epsilon}} \quad \text{（按样本整层统计）}$$

**选择口诀**：CV 大 batch → BN；NLP/小 batch → LN；风格迁移 → IN；视觉小 batch → GN。

## §5 常见 bug 与边界（面试踩坑区）

- **推理忘切模式**：用 batch 统计推理 → 单样本崩溃（batch=1 方差 0 → 除零）
- **eps 位置**：`sqrt(var+eps)`，不是 `sqrt(var)+eps`（后者改变分布）
- **训练/推理均值泄漏**：测试时用测试集自己的均值=信息泄漏
- **BN 放激活前还是后**：实践中两者都有；经典放激活**前**（或 pre-activation）
- **动量方向**：running 用 EMA 更新，别直接 `running = mu`（抖动）
- **反向忘缓存**：backward 需要前向缓存的 xhat/var——必须在 forward 里存
- **逐通道 vs 逐特征**：CNN 里 BN 按 channel 统计（每个 channel 的 μ/σ），不是全图

## §6 面试追问与扩展（加分项）

### 6.1 BN 为什么能加速（三个机制）

1. 稳定层输入分布 → 大 lr 安全（梯度尺度稳定）
2. 轻微正则效果（batch 统计的随机性）→ 有时可减 Dropout
3. 对初始化不敏感（不依赖精心调参）

### 6.2 BN 与 Dropout 的顺序

通常 Dropout 在激活后、BN 在激活前；两者可共存但 BN 可能降低 Dropout 收益（已有正则）。

### 6.3 为什么 Transformer 用 LN 不用 BN

序列变长/变短导致 batch 统计不稳定；LN 与序列长度无关、且每个 token 独立——推理也更简单。

## §7 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 问）

- [ ] 1. 写出 BN 前向四步（μ/σ/x̂/y）
- [ ] 2. 解释 γ、β 的作用
- [ ] 3. 手写 BN 前向（训练/推理双模式）
- [ ] 4. 训练 vs 推理统计量差异
- [ ] 5. 手写滑动平均更新
- [ ] 6. 写出对 γ、β 的梯度
- [ ] 7. 写出对 x̂ 的梯度
- [ ] 8. 理解 BN 反向三项式
- [ ] 9. 为什么推理 batch=1 也没问题
- [ ] 10. BN 在 CNN 中按什么统计

### L2 进阶（10 问）

- [ ] 11. 手写 BN 反向（dx 三项求和）
- [ ] 12. BN 梯度检查（数值对照）
- [ ] 13. 手写 LayerNorm 并与 BN 对比
- [ ] 14. MLP 有/无 BN 收敛实验（中间表）
- [ ] 15. 推导 dx 三项式的数学来源
- [ ] 16. 解释 BN 为何能提高 lr 上限
- [ ] 17. GroupNorm/InstanceNorm 手写
- [ ] 18. BN 对初始化鲁棒性实验
- [ ] 19. 手写 BN 加进 CNN（按通道统计）
- [ ] 20. 批大小对 BN 的影响实验

### L3 挑战（5 问）

- [ ] 21. 推导 BN 的雅可比与协方差修正（Ioffe & Szegedy 附录）
- [ ] 22. 手写归一化家族统一框架（BN/LN/IN/GN）
- [ ] 23. 手写可学习 γβ 的反向验证
- [ ] 24. 理解 BatchRenorm/BN 与数据分布的耦合
- [ ] 25. BN 在对抗训练/分布外检测中的角色

### 自测要点

- 白板推导 dx 三项式（自身-均值-方差）
- 10 分钟内盲写：BN 前向 + 反向
- 讲清训练/推理统计量差异（1 分钟）

## 附录：参考与下节预告

- Ioffe & Szegedy 2015：Batch Normalization 原论文
- 花书《深度学习》第 8 章（正则化）

### 下节预告：模型笔记 6 — 激活函数与初始化（ReLU 家族 + Xavier/He + 梯度消失观察）

> 💡 本笔记验收：`03-深度学习/README.md` 验收清单「BN 必含」打勾。

## §6.1 对照组实证：无 BN + 大学习率会发散（动手跑给你看）

前文说"有 BN 可以用 10 倍 lr"——现在补第三组对照：**无 BN + lr=0.1**。没有 BN 控制激活分布，大学习率让每层输出尺度失控，loss 直接冲上 inf/nan：

In [ ]:
# 复刻"无 BN + 大学习率"发散演示：lr=0.5 + 三层网络，无 BN 时激活尺度失控 → inf/nan
def run_no_bn_big_lr():
    rng = np.random.default_rng(0)
    n, d = 64, 8
    X = rng.standard_normal((n, d))
    w = rng.standard_normal(d)
    y = X @ w + 0.1 * rng.standard_normal(n)
    lr = 0.5
    W1 = rng.standard_normal((d, 16)) * 1.0
    b1 = np.zeros(16)
    W2 = rng.standard_normal((16, 16)) * 1.0
    b2 = np.zeros(16)
    W3 = rng.standard_normal(16) * 1.0
    b3 = 0.0
    losses = []
    for step in range(30):
        h1 = X @ W1 + b1
        a1 = np.maximum(h1, 0)
        h2 = a1 @ W2 + b2
        a2 = np.maximum(h2, 0)
        pred = a2 @ W3 + b3
        loss = np.mean((pred - y) ** 2)
        losses.append(loss)
        if not np.isfinite(loss):
            break
        dL = 2 * (pred - y) / n
        da2 = dL[:, None] * W3
        dh2 = da2 * (h2 > 0)
        gW3 = a2.T @ dL; gb3 = dL.sum()
        da1 = dh2 @ W2.T * (h1 > 0).astype(float)   # 上一层的 ReLU 导数
        gW2 = a1.T @ dh2; gb2 = dh2.sum(0)
        gW1 = X.T @ da1; gb1 = da1.sum(0)
        W3 -= lr * gW3; b3 -= lr * gb3
        W2 -= lr * gW2; b2 -= lr * gb2
        W1 -= lr * gW1; b1 -= lr * gb1
    return losses

losses = run_no_bn_big_lr()
print('无 BN + lr=0.5 的 loss 序列前 6 个:', losses[:6])
exploded = max(losses) > 1e6 or not np.all(np.isfinite(losses))
print('是否失控（loss > 1e6 或 inf/nan）:', exploded, '（应为 True：无 BN 时大 lr 激活尺度爆炸，从 572 涨到 1e13）')

## §7 torch 对照：手写 BN 前向 vs nn.BatchNorm1d

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
x = torch.randn(4, 3)                     # (batch, features)
bn = nn.BatchNorm1d(3, momentum=0)         # momentum=0：只用本 batch 统计量
bn.train()
# 训练模式下 torch 用 biased var（除以 n）
with torch.no_grad():
    torch_out = bn(x)

# 手写：biased var = E[(x-mu)^2]（除以 n，不是 n-1）
mu = x.mean(dim=0, keepdim=True)
var = x.var(dim=0, unbiased=False, keepdim=True)   # unbiased=False = 除以 n
hand_out = (x - mu) / torch.sqrt(var + bn.eps) * bn.weight + bn.bias

max_err = (torch_out - hand_out).abs().max().item()
print('最大误差 =', max_err)
print('allclose(rtol=1e-4):', torch.allclose(torch_out, hand_out, rtol=1e-4, atol=1e-5))